# Lecture 15 - Object-Oriented Programming for Data Science

## Learning Objectives

- Define classes and use __init__ to initialize objects
- Distinguish between instance, class, and static methods
- Implement special methods: __str__, __repr__, __len__, __getitem__
- Use inheritance and super()
- Understand composition over inheritance
- Apply @property for computed attributes

## Key Topics

- class definition and __init__
- Instance vs class vs static methods
- Special methods: __str__, __repr__, __len__, __getitem__
- Inheritance and super()
- Composition over inheritance
- @property decorator

## Class Definition and `__init__`

A **class** is a blueprint for creating objects. The `__init__` method is the constructor — it's called when you create an instance. Inside `__init__`, you define instance attributes that store data unique to each object.

In data science, classes are useful for encapsulating state and behavior together. For example, a `DataFrame` holds data and provides methods like `.mean()` and `.dropna()`. You'll create classes that model datasets, transformers, and models.

In [ ]:
class Dataset:
    def __init__(self, name, data):
        self.name = name
        self.data = data
        self.row_count = len(data)

    def info(self):
        print(f"Dataset '{self.name}': {self.row_count} rows")

# Create instances
ds1 = Dataset("Customers", ["Alice", "Bob", "Charlie"])
ds2 = Dataset("Products", ["Widget", "Gadget"])

ds1.info()
ds2.info()
print(f"ds1 has {ds1.row_count} rows, ds2 has {ds2.row_count} rows")


## Instance vs Class vs Static Methods

- **Instance methods** take `self` and operate on instance data (most common).
- **Class methods** take `cls` and operate on the class itself. Defined with `@classmethod`. Useful for alternative constructors.
- **Static methods** take neither `self` nor `cls`. Defined with `@staticmethod`. They behave like regular functions but live in the class namespace.

Understanding the difference helps you design clean APIs for your data science classes.

In [ ]:
class DataAnalyzer:
    category = "General"  # class attribute

    def __init__(self, values):
        self.values = values

    def range(self):
        return max(self.values) - min(self.values)

    @classmethod
    def from_csv_column(cls, filepath, col_index=0):
        import csv
        values = []
        with open(filepath, "r") as f:
            reader = csv.reader(f)
            next(reader)
            for row in reader:
                try:
                    values.append(float(row[col_index]))
                except (ValueError, IndexError):
                    continue
        return cls(values)

    @staticmethod
    def is_valid(value):
        return value is not None and value >= 0

# Instance method
analyzer = DataAnalyzer([10, 25, 18, 33, 47])
print(f"Range: {analyzer.range()}")

# Class method (alternative constructor)
col_analyzer = DataAnalyzer.from_csv_column("sample_data.csv")
print(f"From CSV — Range: {col_analyzer.range():.2f}")

# Static method (utility)
print(f"Valid? {DataAnalyzer.is_valid(42)}  {DataAnalyzer.is_valid(-1)}  {DataAnalyzer.is_valid(None)}")


## Special Methods: `__str__`, `__repr__`, `__len__`, `__getitem__`

Special methods (dunder methods) let your objects behave like built-in types.

- `__str__`: User-friendly string (used by `print()`).
- `__repr__`: Developer-friendly string (used by `repr()`, debugging).
- `__len__`: Supports `len(obj)`.
- `__getitem__`: Supports `obj[key]` and iteration.

Implementing these makes your classes intuitive to use.

In [ ]:
class TimeSeries:
    def __init__(self, timestamps, values):
        self.timestamps = timestamps
        self.values = values

    def __len__(self):
        return len(self.values)

    def __getitem__(self, index):
        return (self.timestamps[index], self.values[index])

    def __str__(self):
        return f"TimeSeries({len(self)} points: {min(self.values):.1f} to {max(self.values):.1f})"

    def __repr__(self):
        return f"TimeSeries(timestamps={self.timestamps!r}, values={self.values!r})"

ts = TimeSeries(["2024-01-01", "2024-01-02", "2024-01-03"], [22.5, 23.1, 21.8])
print(f"Length: {len(ts)}")
print(f"Item 0: {ts[0]}")
print(f"String: {ts}")
print(f"Repr: {repr(ts)}")


## Inheritance and `super()`

**Inheritance** lets you define a child class that reuses, extends, or overrides behavior from a parent class. Use `super()` to call the parent's methods.

In data science, inheritance is useful for building model families. For example, you might have a `BaseModel` class with a `.train()` and `.predict()` interface, and then `LinearRegression`, `DecisionTree`, and `RandomForest` subclasses that each implement these methods differently.

In [ ]:
class BaseModel:
    def __init__(self, name):
        self.name = name
        self._trained = False

    def fit(self, X, y):
        self._trained = True
        print(f"{self.name}: Model trained on {len(X)} samples.")

    def predict(self, X):
        if not self._trained:
            raise RuntimeError("Model not trained yet!")
        return [0] * len(X)

class LinearRegression(BaseModel):
    def __init__(self):
        super().__init__("LinearRegression")
        self.coefficients = None

    def fit(self, X, y):
        super().fit(X, y)
        import random
        self.coefficients = [random.uniform(-1, 1) for _ in range(len(X[0]))]
        print(f"  Coefficients learned: {[round(c, 3) for c in self.coefficients]}")

class DecisionTree(BaseModel):
    def __init__(self, max_depth=5):
        super().__init__("DecisionTree")
        self.max_depth = max_depth

    def fit(self, X, y):
        super().fit(X, y)
        print(f"  Tree built with max_depth={self.max_depth}")

models = [LinearRegression(), DecisionTree(max_depth=3)]
X = [[1, 2], [3, 4], [5, 6]]
y = [3, 7, 11]
for model in models:
    model.fit(X, y)
    print(f"  Predictions: {model.predict(X)}\n")


## Composition Over Inheritance

**Composition** means building a class by including instances of other classes as attributes. The principle "favor composition over inheritance" reminds you that putting objects together is often more flexible than rigid class hierarchies.

For example, a `Pipeline` class that holds a list of transformer objects (each with `.fit()` and `.transform()` methods) uses composition. You can add, remove, or reorder transformers without changing any class hierarchy.

In [ ]:
class Scaler:
    def fit(self, X):
        self.min_ = min(X)
        self.max_ = max(X)
        return self

    def transform(self, X):
        return [(x - self.min_) / (self.max_ - self.min_) for x in X]

class LogTransformer:
    def transform(self, X):
        import math
        return [math.log(x) for x in X]

# Composition: Pipeline holds transformers
class Pipeline:
    def __init__(self):
        self.steps = []

    def add_step(self, transformer):
        self.steps.append(transformer)
        return self

    def fit_transform(self, X):
        result = X
        for step in self.steps:
            if hasattr(step, "fit"):
                step.fit(result)
            result = step.transform(result)
        return result

pipeline = Pipeline()
pipeline.add_step(Scaler()).add_step(LogTransformer())
result = pipeline.fit_transform([1, 2, 5, 10, 20])
print(f"Pipeline result: {[round(r, 3) for r in result]}")


## The `@property` Decorator

The `@property` decorator lets you define methods that can be accessed like attributes (without parentheses). This is useful for **computed attributes** — values derived from other instance data.

Properties provide a clean interface while keeping the underlying implementation flexible. You can change the computation later without breaking code that uses `.attribute` instead of `.attribute()`.

In [ ]:
class DataFrameSummary:
    def __init__(self, data):
        self._data = data

    @property
    def count(self):
        return len(self._data)

    @property
    def mean(self):
        return sum(self._data) / len(self._data)

    @property
    def range(self):
        return max(self._data) - min(self._data)

    @property
    def summary(self):
        return {
            "count": self.count,
            "mean": round(self.mean, 2),
            "range": round(self.range, 2),
            "min": min(self._data),
            "max": max(self._data),
        }

df = DataFrameSummary([12, 25, 18, 37, 42, 31, 28])
print(f"Mean: {df.mean}")         # Accessed like an attribute
print(f"Count: {df.count}")       # No parentheses needed
print(f"Full summary: {df.summary}")


## Data Science Connection

Object-oriented programming lets you model real-world data science workflows as objects. A `DataFrame` holds tabular data and exposes methods to filter, group, and aggregate. A `Pipeline` chains data transformations. A `Model` base class with subclasses cleanly separates different algorithms. Combined with `@property` for computed attributes and composition for flexible workflows, OOP makes your data science code modular, reusable, and production-ready.